# 05 — Pilotdaten der Zielhardware und kommerzielles Modell

Ersetzt das fruehere Notebook 09 und ergaenzt Teil E (kommerzielles Modell).
Benoetigt die Modelle aus 01 und 04 sowie den Cache aus 04.

| Teil | Frage |
|---|---|
| A | Zero-Shot auf den eigenen Aufnahmen (Huefte, Knoechel rechts/links) |
| B | Betrieb mit der Alarmlogik der Firmware (3 von 5, 0,2 s) |
| C | Einbaulage: Ausrichtung an der Schwerkraft / an den SisFall-Achsen |
| D | Nachtraining auf den Pilotdaten (dreifach, nach Aufnahmen) |
| E | **Kommerzielles Modell**: alle Daten ausser Smart-Insole und MobiAct; Vergleich mit B2r |

Die Sohlenaufnahmen (L_Insole) werden nicht verwendet.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q scipy scikit-learn tensorflow openpyxl

import os, sys, shutil, time, json, pickle, zipfile
from pathlib import Path
import numpy as np
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

# fall_lib laden: zuerst GitHub, sonst Kopie auf Drive. Ohne die
# aktuelle fall_lib waeren alle Ergebnisse falsch (andere Fensterlaenge).
import urllib.request
_LIB_URL = "https://raw.githubusercontent.com/Alyelb/ML-Modell-for-Fall-detection/main/fall_lib.py"
_LIB_DRIVE = "/content/drive/MyDrive/ML_Fall_Detection/fall_lib.py"
try:
    urllib.request.urlretrieve(_LIB_URL, "/content/fall_lib.py")
    print("[fall_lib] von GitHub geladen")
except Exception as _e:
    print(f"[fall_lib] GitHub nicht erreichbar ({_e}) -- nutze Drive-Kopie")
    shutil.copy(_LIB_DRIVE, "/content/fall_lib.py")
sys.path.insert(0, "/content")
import importlib, fall_lib as fl
importlib.reload(fl)
for _f in ("new_run", "build_windows", "znorm", "load_cappimu", "load_pilot",
           "augment_orientation", "rotation_between", "voting_eval", "finetune"):
    assert hasattr(fl, _f), f"fall_lib veraltet ({_f} fehlt) -- neue Version auf Drive legen."
assert fl.WIN_LEN == 100, "falsche Fensterlaenge -- fall_lib pruefen"
print("[fall_lib] Fensterlaenge", fl.WIN_LEN, "bei", fl.FS_TARGET, "Hz")

BASE  = "/content/drive/MyDrive/ML_Fall_Detection"
fl.RESULTS_ROOT = f"{BASE}/results"
MODEL = f"{BASE}/results/fall_model.keras"        # Basismodell (Teil 01)
PILOT = f"{BASE}/Pilotdaten"
CACHE = Path(f"{BASE}/cache"); CACHE.mkdir(exist_ok=True)


def hole(name, ziel):
    """Kopiert einen Datensatz von Drive auf die lokale Platte
    (Drive ist fuer tausende Kleindateien zu langsam). Gibt None
    zurueck, wenn der Ordner fehlt."""
    src = f"{BASE}/{name}"
    if Path(ziel).exists():
        print(f"[ok] {name} bereits lokal"); return ziel
    if not Path(src).exists():
        print(f"[!] {src} nicht gefunden. Vorhanden:")
        for p in sorted(Path(BASE).iterdir()):
            print("   ", p.name)
        return None
    t = time.time(); shutil.copytree(src, ziel)
    print(f"[ok] {name} kopiert in {time.time()-t:.0f}s")
    return ziel


def cached(name, fn):
    """Geladene Trials auf Drive zwischenspeichern (Laden dauert lange)."""
    p = CACHE / f"{name}.pkl"
    if p.exists():
        print(f"[cache] {name}")
        return pickle.load(open(p, "rb"))
    r = fn(); pickle.dump(r, open(p, "wb")); return r


from tensorflow import keras
def lade_modell(pfad):
    if not Path(pfad).exists():
        raise FileNotFoundError(f"{pfad} fehlt -- vorheriges Notebook ausfuehren.")
    m = keras.models.load_model(pfad, compile=False)
    m.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
    return m

print("Konfiguration:", fl.default_config())

In [ ]:
MODELLE = {"Basis": lade_modell(f"{BASE}/results/fall_model.keras"),
           "Lower-Limb": lade_modell(f"{BASE}/results/fall_model_lowerlimb.keras")}
_aug = Path(f"{BASE}/results/fall_model_lowerlimb_aug.keras")   # aus 04, Teil D
if _aug.exists():
    MODELLE["LL-aug (B2r)"] = lade_modell(str(_aug))
run = fl.new_run("09_pilotdaten")
ergebnisse = {}

## Aufnahmen

In [ ]:
alle = [t for t in fl.load_pilot(PILOT) if "INSOLE" not in t["position"]]   # Sohle ausgeschlossen
POS = sorted({t["position"] for t in alle})
for p in POS:
    for f in sorted({t["datei"] for t in alle if t["position"] == p and t["label"]}):
        n = sum(1 for t in alle if t["position"] == p and t["datei"] == f)
        print(f"  {p:<12} {f:<34} {n} Aufprall(e)")

## Teil A — Zero-Shot (aufprallzentriertes Fenster)

In [ ]:
def vorhersage(m, X):
    return m.predict(fl.znorm(X), batch_size=512, verbose=0).ravel()

teilA = {}
print(f"{'Position':<13}{'Modell':<12}{'Se':>7}{'95%-KI':>17}{'Sp':>7}")
for p in POS:
    X, y, _, _ = fl.build_windows([t for t in alle if t["position"] == p], mode="standard")
    for name, m in MODELLE.items():
        pr = vorhersage(m, X)
        tp = int(((pr > .5) & (y == 1)).sum()); fn = int(((pr <= .5) & (y == 1)).sum())
        tn = int(((pr <= .5) & (y == 0)).sum()); fp = int(((pr > .5) & (y == 0)).sum())
        ci = fl.wilson_ci(tp, tp + fn)
        teilA[f"{p}|{name}"] = {"Se": tp / (tp + fn), "Se_ci95": ci, "Sp": tn / (tn + fp),
                                "TP": tp, "FN": fn, "FP": fp, "TN": tn}
        print(f"{p:<13}{name:<12}{tp/(tp+fn):>7.2f}   [{ci[0]:.2f}; {ci[1]:.2f}]{tn/(tn+fp):>7.2f}")
ergebnisse["teilA_zero_shot"] = teilA

## Teil B — Betrieb mit Alarmlogik

Wie in der Firmware: Inferenz alle 0,2 s, Alarm bei 3 positiven von 5
Entscheidungen (Schwelle 0,5). Ein Sturz gilt als erkannt, wenn bis
1 s nach dem Aufprall ein Alarm ausgelöst wird. Gezählt werden
Fehlalarm-**Ereignisse** in den Alltagsaufnahmen.

In [ ]:
SCHRITT = int(0.2 * fl.FS_TARGET)

def folgen(m, trials, R=None):
    seqs = []
    for t in trials:
        x = t["data"] if R is None else fl.rotate_windows(t["data"][None], R)[0]
        W = np.asarray(fl.window_sliding(x, fl.WIN_LEN, SCHRITT))
        if not len(W):
            continue
        p = vorhersage(m, W)
        if t["label"]:
            c = t["impact_idx"]
            lo = max(0, (c - fl.WIN_LEN) // SCHRITT)
            seqs.append((p[lo:c // SCHRITT + 6], True))
        else:
            seqs.append((p, False))
    return seqs

def betrieb(m, trials, R=None):
    r = fl.voting_eval(folgen(m, trials, R), 0.5, 3, 5, SCHRITT / fl.FS_TARGET)
    r1 = fl.voting_eval(folgen(m, trials, R), 0.5, 1, 1, SCHRITT / fl.FS_TARGET)
    return {"Se_3v5": r["Se"], "fp_3v5": r["fp_events"], "Se_1v1": r1["Se"],
            "fp_1v1": r1["fp_events"], "adl_min": round(r["adl_hours"] * 60, 1)}

# Feste Einbaulage der Firmware (modell.h, EINBAU_SOHLE): Sohle flach,
# Bauteilseite oben, USB-C zur Aussenseite -> Achsrahmen Knoechel
EINBAU = {p: np.array([[1, 0, 0], [0, 0, -1], [0, 1, 0]], float)
          for p in {t["position"] for t in alle} if "INSOLE" in p}

teilB = {}
print(f"{'Position':<13}{'Modell':<12}{'Se 1v1':>8}{'Se 3v5':>8}{'FP-Ereign. 3v5':>16}{'ADL min':>9}")
for p in POS:
    tr = [t for t in alle if t["position"] == p]
    for name, m in MODELLE.items():
        r = betrieb(m, tr); teilB[f"{p}|{name}"] = r
        print(f"{p:<13}{name:<12}{r['Se_1v1']:>8.2f}{r['Se_3v5']:>8.2f}{r['fp_3v5']:>16}{r['adl_min']:>9}")
        if p in EINBAU:
            r = betrieb(m, tr, EINBAU[p]); teilB[f"{p}+Einbau|{name}"] = r
            print(f"{p+'+Einbau':<13}{name:<12}{r['Se_1v1']:>8.2f}{r['Se_3v5']:>8.2f}{r['fp_3v5']:>16}{r['adl_min']:>9}")
ergebnisse["teilB_betrieb"] = teilB

## Teil C — Einbaulage

Jedes Modell erwartet die Achslage seines Trainingsdatensatzes. Hier
wird die mittlere Schwerkraftrichtung im Stehen auf die des
Trainingsdatensatzes gedreht (kleinste Drehung). Die Drehung um die
Schwerkraftachse bleibt dabei unbestimmt. Verbessert sich das Ergebnis
deutlich, erklärt die Einbaulage einen Teil der Domänenlücke.

Referenz Basis: SisFall-Alltagsaufnahmen. Referenz Lower-Limb:
CAPPIMU-Schienbein (Cache aus Notebook 10).

In [ ]:
REF = {}
sis_lokal = "/content/SisFall"
if not Path(sis_lokal).exists():
    shutil.copytree(f"{BASE}/SisFall_dataset", sis_lokal)
REF["Basis"] = fl.mean_gravity(fl.load_sisfall(sis_lokal, verbose=False))
knie = pickle.load(open(CACHE / "cappimu_knee_r.pkl", "rb")) + \
       pickle.load(open(CACHE / "cappimu_knee_l.pkl", "rb"))
REF["Lower-Limb"] = fl.mean_gravity(knie); del knie
print("Schwerkraft Referenz:", {k: np.round(v, 2).tolist() for k, v in REF.items()})

teilC = {}
print(f"\n{'Position':<13}{'Modell':<12}{'Winkel':>8}{'Se 3v5 vorher':>15}{'nachher':>9}{'FP vorher':>11}{'nachher':>9}")
for p in POS:
    tr = [t for t in alle if t["position"] == p]
    g = fl.mean_gravity(tr)
    for name, m in MODELLE.items():
        R = fl.rotation_between(g, REF[name])
        w = float(np.degrees(np.arccos(np.clip(g @ REF[name], -1, 1))))
        v, n = teilB[f"{p}|{name}"], betrieb(m, tr, R)
        teilC[f"{p}|{name}"] = {"winkel_grad": round(w), "nachher": n, "R": R.round(4).tolist()}
        print(f"{p:<13}{name:<12}{w:>7.0f}°{v['Se_3v5']:>15.2f}{n['Se_3v5']:>9.2f}"
              f"{v['fp_3v5']:>11}{n['fp_3v5']:>9}")
ergebnisse["teilC_einbaulage"] = teilC

## Teil C2 — Hüfte in den SisFall-Achsrahmen

SisFall (Gürtelschnalle): **+z vorwärts, +y in Schwerkraftrichtung (nach unten), +x nach rechts**
(Sucerquia et al. 2017). Ein Beschleunigungssensor misst im Stand +1 g auf der Achse, die nach **oben**
zeigt; in SisFall also y ≈ −1 g.

Eigener Prototyp an der Hüfte: im Stand x ≈ −0,9 g → **+x zeigt nach unten**. Liegelage nach
Vorwärtssturz z ≈ −0,9 g (Rücken oben), nach Rückwärtssturz z ≈ +0,8 g → **+z zeigt nach vorn**.
Damit ist rechts = unten × vorn = −y. Feste Drehung: x_S = −y, y_S = x, z_S = z, danach Ausgleich
der Neigung im Stand (Platine ca. 25° gekippt). Alles aus der Einbaulage, nicht aus den Ergebnissen.

In [ ]:
R_GEO = np.array([[0, -1, 0], [1, 0, 0], [0, 0, 1]], float)

def sisfall_ausrichtung(trials):
    g = fl.mean_gravity(trials)                       # Stand, eigener Sensor
    # Kontrolle der Vorwaertsrichtung aus den Liegephasen
    for art in ("fall_forward", "fall_backward"):
        L = [t["data"][t["impact_idx"] + 75:t["impact_idx"] + 250, :3].mean(0)
             for t in trials if t["label"] and t["activity"] == art]
        if L:
            print(f"  Liegelage {art:<14} g = {np.round(np.mean(L, 0), 2)}")
    return fl.rotation_between(R_GEO @ g, np.array([0., -1., 0.])) @ R_GEO

tw = [t for t in alle if t["position"] == "WAIST"]
if tw:
    R_W = sisfall_ausrichtung(tw)
    print("R (Huefte -> SisFall):\n", np.round(R_W, 2))
    teilC2 = {}
    for name, R in [("roh", None), ("nur Achsen", R_GEO), ("Achsen + Neigung", R_W)]:
        r = fl.voting_eval(folgen(MODELLE["Basis"], tw, R), 0.5, 3, 5, 0.2)
        teilC2[name] = r
        print(f"  Basis {name:<18} Se {r['detected']}/{r['total_falls']}  FP-Ereignisse {r['fp_events']}")
    ergebnisse["teilC2_huefte_sisfall"] = teilC2

## Teil D — Nachtraining auf den Pilotdaten

Ausgangspunkt ist das Lower-Limb-Modell. Dreifach kreuzvalidiert: Die
Sturzdateien werden nach ihrer Nummer (00/01/02) und die
Alltagsaufnahmen nach Dritteln ihrer Dauer auf die Falten verteilt; je
Falte wird auf zwei Dritteln trainiert und auf dem dritten bewertet.
Positionen mit nur einer Sturzdatei je Sturzart werden nach Sturzart
aufgeteilt. Für die Sohle wird vorher die feste Einbaulage der Firmware
angewendet.

In [ ]:
def falten_zuordnen(trials, k=3):
    nummern = sorted({t["datei"][:2] for t in trials if t["label"]})
    nach_nummer = len(nummern) >= k
    arten = sorted({t["activity"] for t in trials if t["label"]})
    teile = []
    for t in trials:
        if t["label"]:
            g = (nummern.index(t["datei"][:2]) if nach_nummer
                 else arten.index(t["activity"])) % k
            teile.append((g, t))
        else:
            n = len(t["data"])
            for j in range(k):
                seg = t["data"][j * n // k:(j + 1) * n // k]
                if len(seg) >= fl.WIN_LEN:
                    teile.append((j, {**t, "data": seg}))
    return teile

teilD = {}
for p in POS:
    tr = [t for t in alle if t["position"] == p]
    if p in EINBAU:
        tr = [{**t, "data": fl.rotate_windows(t["data"][None], EINBAU[p])[0]} for t in tr]
    teile = falten_zuordnen(tr)
    seq_vor, seq_nach = [], []
    for g in range(3):
        train = [t for f, t in teile if f != g]; test = [t for f, t in teile if f == g]
        if not any(t["label"] for t in test):
            continue
        Xf = [fl.window_impact_centered(t["data"], impact_idx=t["impact_idx"])[0]
              for t in train if t["label"]]
        Xa = [w for t in train if not t["label"] for w in fl.window_sliding(t["data"], fl.WIN_LEN, 25)]
        X = np.asarray(Xf + Xa, np.float32); y = np.r_[np.ones(len(Xf)), np.zeros(len(Xa))]
        m = fl.finetune(MODELLE["Lower-Limb"], fl.znorm(X), y)
        seq_vor += folgen(MODELLE["Lower-Limb"], test); seq_nach += folgen(m, test)
    vor = fl.voting_eval(seq_vor, 0.5, 3, 5, 0.2); nach = fl.voting_eval(seq_nach, 0.5, 3, 5, 0.2)
    teilD[p] = {"vorher": vor, "nachher": nach}
    print(f"{p:<13} vorher Se {vor['Se']:.2f}, FP-Ereign. {vor['fp_events']:>3} | "
          f"nachher Se {nach['Se']:.2f}, FP-Ereign. {nach['fp_events']:>3}  "
          f"({vor['adl_hours']*60:.1f} min ADL)")
ergebnisse["teilD_nachtraining"] = teilD

## Speichern

In [ ]:
fl.save_results(run, ergebnisse, config=fl.default_config(),
    notes=("Pilotdaten: Zero-Shot, Betrieb 3v5, Einbaulage (Schwerkraft), "
           "SisFall-Achsen Huefte, Nachtraining 3-fach. Eine Person, ohne Sohle."))

# Teil E — Kommerzielles Modell fuer Clever-Sole

Trainiert auf allen verfuegbaren Datensaetzen **ausser Smart-Insole und MobiAct**
(nicht-kommerzielle Lizenz). Ablauf:

1. **Vortraining** von Grund auf auf einem gemischten Pool aller Positionen
   (SisFall, KFall, UMAFall, FallAllD, WEDA, CAPPIMU ohne Schienbein),
   orientierungsaugmentiert.
2. **Kreuzvalidierter Vergleich** mit dem Berichtsmodell B2r auf identischen
   Falten (CAPPIMU-Schienbein, ImbalanSense). Smart-Insole und Pilotdaten sind
   fuer beide Modelle extern. Der Unterschied prueft die Behauptung, dass
   breiteres Vortraining dem Fussmodell hilft.
3. **Endmodell** auf allen Daten inkl. eigener Pilotaufnahmen, int8-Export.

> Lizenzen aller Datensaetze vor der Uebergabe pruefen (Schalter `DATEN`).

In [ ]:
# ---------------------------------------------------------------
# Teil E: Kommerzielles Modell -- alle Daten ausser Smart-Insole und MobiAct
# ---------------------------------------------------------------
# Lizenzen VOR der Uebergabe pruefen. Ein Datensatz, der sich als nicht
# kommerziell nutzbar herausstellt, wird hier auf False gesetzt und das
# Notebook ab dieser Zelle neu ausgefuehrt.
DATEN = {"SisFall": True, "KFall": True, "UMAFall": True, "FallAllD": True,
         "WEDA": True, "CAPPIMU": True, "ImbalanSense": True, "Pilot": True}
MAX_ADL_JE_SATZ = 15000
rng = np.random.default_rng(7)
run_k = fl.new_run("13_kommerziell")

def kappe_adl(X, y, n):
    f = np.where(y == 1)[0]; a = np.where(y == 0)[0]
    if len(a) > n:
        a = rng.choice(a, n, replace=False)
    i = np.r_[f, a]
    return X[i], y[i]

pool = {}
def dazu(name, trials):
    X, y, g, _ = fl.build_windows(trials, mode="standard")
    X, y = kappe_adl(X, y, MAX_ADL_JE_SATZ)
    pool[name] = (X, y)
    print(f"  {name:<18} {len(y):>7} Fenster, {int(y.sum()):>6} Stuerze")

if DATEN["SisFall"]:
    dazu("SisFall", cached("sisfall", lambda: fl.load_sisfall(hole("SisFall_dataset", "/content/SisFall"), verbose=False)))
if DATEN["KFall"]:
    dazu("KFall", cached("kfall", lambda: fl.load_kfall(hole("KFALL/sensor_data_new", "/content/KFall"),
         label_dir=hole("KFALL/label_data_new", "/content/KFall_labels"), verbose=False)))
if DATEN["UMAFall"]:
    UMA = hole("UMAFall Fall Detection/UMAFall_Dataset_corrected", "/content/UMAFall")
    for pos in fl.UMAFALL_POSITIONS:
        dazu(f"UMAFall {pos}", cached(f"umafall_{pos}", lambda: fl.load_umafall(UMA, position=pos, verbose=False)))
if DATEN["FallAllD"]:
    FA = hole("FallAllD", "/content/FallAllD")
    for dev in fl.FALLALLD_DEVICES:
        dazu(f"FallAllD {fl.FALLALLD_DEVICES[dev]}",
             cached(f"fallalld_{dev}", lambda: fl.load_fallalld(FA, device=dev, verbose=False)))
if DATEN["WEDA"]:
    W50 = "/content/WEDA50"
    if not Path(W50).exists():
        shutil.copytree(f"{BASE}/WEDA-FALL-main/dataset/50Hz", W50)
    dazu("WEDA", cached("weda_alle", lambda: fl.load_weda(W50, verbose=False)))
CAP_ROOT = next(Path(f"{BASE}/CAPPIMU Dataset").rglob("subject_1")).parent
if DATEN["CAPPIMU"]:
    # Rumpf- und Armpositionen fuer das Vortraining; Schienbein bleibt fuer
    # das Nachtraining und die Kreuzvalidierung reserviert
    for pos in ["head", "chest", "arm_l", "arm_r", "wrist_l", "wrist_r", "pocket"]:
        dazu(f"CAPPIMU {pos}", cached(f"cappimu_{pos}", lambda: fl.load_cappimu(CAP_ROOT, pos)))

In [ ]:
# Vortraining von Grund auf, orientierungsaugmentiert (Positionen gemischt)
X = np.concatenate([v[0] for v in pool.values()]); y = np.concatenate([v[1] for v in pool.values()])
del pool
X, y = fl.augment_orientation(X, y, copies=1, seed=1)
p = rng.permutation(len(y)); X = fl.znorm(X[p]); y = y[p]
print(f"Vortraining: {len(y)} Fenster ({int(y.sum())} Stuerze)")
t = time.time()
m_pool = fl.train(fl.build_cnn(), X, y, epochs=30, verbose=1)
print(f"  fertig in {time.time()-t:.0f}s")
del X, y; import gc; gc.collect()
m_pool.save(f"{BASE}/results/fall_model_pool.keras")

## Kreuzvalidierter Vergleich mit B2r

In [ ]:
# Vergleich mit dem Berichtsmodell B2r auf denselben Falten.
# B2r: SisFall -> KFall -> Schienbein + ImbalanSense (augmentiert)
# KOM: breites Vortraining (alle Datensaetze) -> Schienbein + ImbalanSense (augmentiert)
# Smart-Insole und Pilotdaten sind fuer beide extern.
m_wk = lade_modell(f"{BASE}/results/fall_model_kfall.keras")
knie = cached("cappimu_knee_l", lambda: fl.load_cappimu(CAP_ROOT, "knee_l")) + \
       cached("cappimu_knee_r", lambda: fl.load_cappimu(CAP_ROOT, "knee_r"))
Xc, yc, gc_, _ = fl.build_windows(knie, mode="standard"); del knie
def aus_cache(name):
    p = CACHE / f"{name}.pkl"
    if not p.exists():
        raise FileNotFoundError(f"{p} fehlt -- bitte zuerst 04_fuss_orientierung ausfuehren.")
    return pickle.load(open(p, "rb"))
Xi, yi, gi, _ = fl.build_windows(aus_cache("imbalansense"), mode="standard")
Xs, ys, _, _ = fl.build_windows(aus_cache("smart_insole"), mode="standard")
PIL = {p: fl.build_windows([t for t in alle if t["position"] == p], mode="standard") for p in POS}

def vorhersage(m, X, bs=4096):
    return np.concatenate([m.predict(fl.znorm(X[i:i+bs]), batch_size=512, verbose=0).ravel()
                           for i in range(0, len(X), bs)]) if len(X) else np.empty(0)

def kennzahlen(p, y, thr=0.5):
    tp = int(((p > thr) & (y == 1)).sum()); fn = int(((p <= thr) & (y == 1)).sum())
    fp = int(((p > thr) & (y == 0)).sum()); tn = int(((p <= thr) & (y == 0)).sum())
    h = (fp + tn) * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
    return {"Se": tp / (tp + fn) if tp + fn else None, "Sp": tn / (tn + fp) if tn + fp else None,
            "Acc": (tp + tn) / max(tp + tn + fp + fn, 1), "fp_per_hour": fp / h if h else None,
            "TP": tp, "FN": fn, "FP": fp, "TN": tn}

ROT = fl.random_rotation_matrices(5, seed=7)
def bewerte(m, te_c, te_i):
    r = {"knie": kennzahlen(vorhersage(m, Xc[te_c]), yc[te_c]),
         "imb": kennzahlen(vorhersage(m, Xi[te_i]), yi[te_i]),
         "si": kennzahlen(vorhersage(m, Xs), ys)}
    f = te_c[yc[te_c] == 1]
    r["knie_rot_Se"] = float(np.mean([kennzahlen(vorhersage(m, fl.rotate_windows(Xc[f], R)), yc[f])["Se"]
                                      for R in ROT]))
    for p, (X, y, _, _) in PIL.items():
        r[f"pilot_{p}"] = kennzahlen(vorhersage(m, X), y)
    return r

def nachtrain_satz(tr_c, tr_i, seed):
    f = tr_c[yc[tr_c] == 1]; a = tr_c[yc[tr_c] == 0]
    a = rng.choice(a, min(20000, len(a)), replace=False)
    ii = rng.choice(tr_i, min(10000, len(tr_i)), replace=False)
    X = np.concatenate([Xc[f], Xc[a], Xi[ii]]); y = np.concatenate([yc[f], yc[a], yi[ii]])
    X, y = fl.augment_orientation(X, y, copies=1, seed=seed)
    p = rng.permutation(len(y))
    return fl.znorm(X[p]), y[p]

N_FOLDS = 5
sp_c = fl.group_kfold_splits(gc_, yc, n_splits=N_FOLDS)
sp_i = fl.group_kfold_splits(gi, None, n_splits=N_FOLDS)
kv = {"B2r": [], "KOM": []}
for i, ((tr_c, te_c), (tr_i, te_i)) in enumerate(zip(sp_c, sp_i), 1):
    t = time.time()
    X, y = nachtrain_satz(tr_c, tr_i, seed=i)
    for k, start in [("B2r", m_wk), ("KOM", m_pool)]:
        m = fl.finetune(start, X, y, freeze=0)
        kv[k].append(bewerte(m, te_c, te_i)); del m
    del X
    print(f"Falte {i}: " + " | ".join(
        f"{k} Knie Se {kv[k][-1]['knie']['Se']:.3f} (gedreht {kv[k][-1]['knie_rot_Se']:.3f}), "
        f"SI FP/h {kv[k][-1]['si']['fp_per_hour']:.0f}" for k in kv) + f"  ({time.time()-t:.0f}s)")

In [ ]:
def pool_k(L, s):
    tp = sum(r[s]["TP"] for r in L); fn = sum(r[s]["FN"] for r in L)
    fp = sum(r[s]["FP"] for r in L); tn = sum(r[s]["TN"] for r in L)
    h = (fp + tn) * fl.WIN_SEC * (1 - fl.OVERLAP) / 3600
    return {"Se": tp / (tp + fn) if tp + fn else None, "Sp": tn / (tn + fp) if tn + fp else None,
            "Acc": (tp + tn) / max(tp + tn + fp + fn, 1), "fp_per_hour": fp / h if h else None}

tabK = {}
for k in kv:
    z = {s: pool_k(kv[k], s) for s in ["knie", "imb", "si"] + [f"pilot_{p}" for p in PIL]}
    z["knie_rot_Se"] = float(np.mean([r["knie_rot_Se"] for r in kv[k]]))
    tabK[k] = z
    print(f"{k:<4} Knie Se {z['knie']['Se']:.3f} Acc {z['knie']['Acc']:.3f} gedreht {z['knie_rot_Se']:.3f} | "
          f"IMB FP/h {z['imb']['fp_per_hour']:.1f} | SI FP/h {z['si']['fp_per_hour']:.1f} | "
          + " ".join(f"{p} Se {z[f'pilot_{p}']['Se']:.2f}" for p in PIL))
vglK = {}
for s, met in [("knie", "Se"), ("knie", "Sp"), ("imb", "Sp"), ("si", "Sp")]:
    print(f"\nKOM vs B2r — {s} {met}")
    vglK[f"KOM_vs_B2r_{s}_{met}"] = fl.paired_comparison(
        [r[s] for r in kv["B2r"]], [r[s] for r in kv["KOM"]], "B2r", "KOM", metric=met)
print("\nKOM vs B2r — Schienbein gedreht Se")
vglK["KOM_vs_B2r_knie_rot_Se"] = fl.paired_comparison(
    [{"Se": r["knie_rot_Se"]} for r in kv["B2r"]], [{"Se": r["knie_rot_Se"]} for r in kv["KOM"]],
    "B2r", "KOM", metric="Se")

## Endmodell und Export

In [ ]:
# Endmodell fuer Clever-Sole: alle Schienbein- und ImbalanSense-Daten,
# zusaetzlich die eigenen Pilotaufnahmen (Huefte und Knoechel)
X, y = nachtrain_satz(np.arange(len(yc)), np.arange(len(yi)), seed=99)
if DATEN["Pilot"]:
    Xp = np.concatenate([v[0] for v in PIL.values()]); yp = np.concatenate([v[1] for v in PIL.values()])
    Xp, yp = fl.augment_orientation(Xp, yp, copies=3, seed=5)
    X = np.concatenate([X, fl.znorm(Xp)]); y = np.concatenate([y, yp])
    p = rng.permutation(len(y)); X, y = X[p], y[p]
m_kom = fl.finetune(m_pool, X, y, freeze=0)
pfad = f"{BASE}/results/fall_model_kommerziell.keras"
m_kom.save(pfad); shutil.copy(pfad, run_k / "fall_model_kommerziell.keras")
kb = fl.export_tflite_int8(m_kom, X[:2000], run_k / "fall_model_kommerziell_int8.tflite")
print("gesichert:", pfad)

fl.save_results(run_k, {"tabelle": tabK, "vergleiche": vglK, "falten": kv, "tflite_kb": round(kb, 1),
                        "daten": {k: v for k, v in DATEN.items()}},
    config={**fl.default_config(), "n_folds": N_FOLDS, "max_adl_je_satz": MAX_ADL_JE_SATZ},
    notes=("Kommerzielles Modell: alle Datensaetze ausser Smart-Insole und MobiAct. "
           "CV-Vergleich mit B2r auf identischen Falten. Lizenzen vor Uebergabe pruefen."))